In [ ]:
%load_ext autoreload
%autoreload 2

from spatial_tcr.utils import switch_cwd_to_root

switch_cwd_to_root()

import scanpy as sc

from spatial_tcr.annotation import (
    annotate,
)
from spatial_tcr.tcr import aggregate_trv_expression, get_tcr_genes

In [ ]:
adata = sc.read_h5ad("data/xenium/processed/02-kidney_tcr_split.h5ad")
adata, adata.X.max()

## prep training data

In [ ]:
ad_ref = sc.read(
    "/bonn-epyc/projects/robin/singlecell/kidney/Lake/cellxgene_obj.h5ad"
).raw.to_adata()
ad_ref.var["feature_name"] = ad_ref.var["feature_name"].astype(str)
ad_ref.var = ad_ref.var.set_index("feature_name")
ad_ref

In [ ]:
ad_ref.X.max()

## Prepare annotations

In [ ]:
ad_ref.obs["subclass.l1"].unique().tolist()

In [ ]:
ad_ref.obs["cell_type_new"] = ad_ref.obs["subclass.l1"].astype(str).copy()
mask_immune = ad_ref.obs["cell_type_new"] == "IMM"
ad_ref.obs.loc[mask_immune, "cell_type_new"] = ad_ref.obs.loc[
    mask_immune, "subclass.l2"
]
mask_corpuscle = ad_ref.obs["structure"] == "renal corpuscle"
ad_ref.obs.loc[mask_corpuscle, "cell_type_new"] = ad_ref.obs.loc[
    mask_corpuscle, "subclass.l2"
].astype(str)

# Define the mapping dictionary
ct_map = {
    "dPOD": "podocyte",
    "POD": "podocyte",
    "EC-GC": "glomerular endothelial cell",
    "MC": "mesangial cell",
    "PEC": "parietal epithelial cell",
    # "glomerular endothelial cell": "endothelial cell",
    "MAC-M2": "Mac",
    "ncMON": "Mac",
}


ct_map = {
    "dPOD": "POD",
    "EC-GC": "glom. EC",
    "glomerular endothelial cell": "glom. EC",
    "MAC-M2": "Mac",
    "ncMON": "Mac",
    "cycNKC/T": "T",
    "NKC/T": "T",
}

ad_ref.obs["cell_type_new"] = ad_ref.obs["cell_type_new"].replace(ct_map)

In [ ]:
ad_ref.obs.cell_type_new.value_counts()

In [ ]:
# check tcr
ad_tmp = aggregate_trv_expression(ad_ref)

In [ ]:
markers = ["TRAV", "TRBV", "TRDV", "TRGV"]

dp = sc.pl.dotplot(
    ad_tmp,
    markers,
    groupby="cell_type_new",
    return_fig=True,
    # standard_scale="var",
    cmap="Reds",
)
dp.add_totals().style(dot_edge_color="black", dot_edge_lw=0.5).show()

In [ ]:
markers = ["CD3D", "CD3E", "CD8A", "CD4", "GZMA"]
dp = sc.pl.dotplot(
    ad_ref,
    groupby="cell_type_new",
    var_names=markers,
    return_fig=True,
    dendrogram=False,
)
dp.add_totals().style(dot_edge_color="black", dot_edge_lw=0.5).show()

In [ ]:
ad_ref.obs["cell_type"] = ad_ref.obs["cell_type"].str.replace("kidney", "").str.strip()

## Annotation transfer without tcr genes

In [ ]:
common_genes = ad_ref.var_names.intersection(adata.var_names)

av_genes, bv_genes, dv_genes, gv_genes, tv_genes = get_tcr_genes(adata)

common_genes = [g for g in common_genes if g not in tv_genes]
len(common_genes)

In [ ]:
annotate(
    adata,
    ad_ref,
    label_key="cell_type_new",
    label_out="cell_type_no_tcr",
    genes=common_genes,
)

In [ ]:
markers = [
    # "LRP2",
    "VCAM1",
    # "CLDN1",
    "UMOD",
    "TACSTD2",
    # "SLC12A3",
    # "CALB1",
    "AQP2",
    "ATP6V0D2",
    # "NRXN1",
    "PODXL",
    "PECAM1",
    # "PDGFRB",
    "COL1A1",
    "PTPRC",
    "MS4A1",
    "CD3D",
    "CD3E",
    "CD8A",
    "GZMA",
    "FCGR3A",
    "CD14",
    "CD163",
    # "CLEC4C",
    # "ITGAX",
    # "CLEC9A",
    # "S100A9",
    "MS4A2",
    "CD19",
]
dp = sc.pl.dotplot(
    adata,
    groupby="cell_type_no_tcr",
    var_names=markers,
    return_fig=True,
    dendrogram=False,
)
dp.add_totals().style(dot_edge_color="black", dot_edge_lw=0.5).show()

In [ ]:
adata.obs.cell_type_no_tcr_prob.hist(bins=100)

In [ ]:
# require 75% probability
mask = adata.obs.cell_type_no_tcr_prob > 0.50
ad_bad = adata[~mask].copy()
ad_bad.obs["cell_type_no_tcr"].value_counts()

In [ ]:
# remove unknown cells
adata = adata[adata.obs.cell_type_no_tcr != "unknown"].copy()
adata.obs["cell_type_no_tcr"].value_counts()

In [ ]:
adata

In [ ]:
adata.write_h5ad("data/xenium/processed/03-kidney_tcr_classified.h5ad")